# Salting  
**Salting** is a technique to mitigate skewness by adding a random prefix/suffix to keys, converting a single skewed key into multiple pseudo-keys. This:
- Splits large partitions into smaller chunks.
- Ensures even data distribution across nodes.
- Is commonly used for skewed joins/aggregations.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from functools import reduce

In [4]:
spark = SparkSession.builder.master("local[*]")\
        .appName("saltingApplicationTest")\
        .config("spark.sql.autoBroadcastJoinThreshold", "0B")\
        .config("spark.sql.adaptive.enabled", "false")\
        .getOrCreate()
spark

In [5]:
# --------------------------
# 1. Create skewed large DataFrame
# customer_id = 1 appears many times (simulate skew)
# --------------------------
skewed_data = [(1, f"order_{i}") for i in range(1000000)] + \
              [(i, f"order_{i}") for i in range(2, 100)]

large_df = spark.createDataFrame(skewed_data, ["customer_id", "order_info"])

In [6]:
large_df.show(5)

+-----------+----------+
|customer_id|order_info|
+-----------+----------+
|          1|   order_0|
|          1|   order_1|
|          1|   order_2|
|          1|   order_3|
|          1|   order_4|
+-----------+----------+
only showing top 5 rows



In [7]:
# --------------------------
# 2. Create small lookup DataFrame
# One record per customer
# --------------------------
small_data = [(i, f"tier_{i}") for i in range(1, 100)]
small_df = spark.createDataFrame(small_data, ["customer_id", "membership"])

In [8]:
small_df.show(5)

+-----------+----------+
|customer_id|membership|
+-----------+----------+
|          1|    tier_1|
|          2|    tier_2|
|          3|    tier_3|
|          4|    tier_4|
|          5|    tier_5|
+-----------+----------+
only showing top 5 rows



In [9]:
# joining without salting
large_df.join(small_df, large_df["customer_id"]==small_df["customer_id"], "inner").count()

1000098

In [10]:
# --------------------------
# 3. Add salt to skewed (large) DataFrame
# --------------------------
NUM_SALTS = 100

salted_large_df = large_df.withColumn("salt", floor(rand() * NUM_SALTS)) \
    .withColumn("customer_id_salted", concat_ws("_", large_df.customer_id.cast("string"), 
                                                 floor(rand() * NUM_SALTS).cast("string")))


In [11]:
salted_large_df.show(5)

+-----------+----------+----+------------------+
|customer_id|order_info|salt|customer_id_salted|
+-----------+----------+----+------------------+
|          1|   order_0|  73|              1_23|
|          1|   order_1|  24|              1_31|
|          1|   order_2|  89|              1_90|
|          1|   order_3|  82|              1_25|
|          1|   order_4|  98|              1_67|
+-----------+----------+----+------------------+
only showing top 5 rows



In [12]:
# Create an array of salt values: [0, 1, 2, ..., NUM_SALTS-1]
salt_values = array([lit(i) for i in range(NUM_SALTS)])


In [13]:
# Add salt column by exploding the salt array
salted_small_df = small_df.withColumn("salt", explode(salt_values))

In [14]:
# Generate salted join key
salted_small_df = salted_small_df.withColumn(
    "customer_id_salted",
    concat_ws("_", col("customer_id").cast("string"), col("salt").cast("string"))
)

In [15]:
joined_df = salted_large_df.join(salted_small_df, on="customer_id_salted", how="inner")

In [16]:
joined_df.count()

1000098